In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Protocol, Callable, Optional, Tuple
import random
import statistics

"""
PEAS specification for the vacuum world simulator:

P (Performance measure): cumulative score where each time step the
    environment gives +1 per clean square and subtracts 1 per action.
    The simulator tracks `cumulative_score` in `PerformanceEvaluator`.

E (Environment): explicit `VacuumEnvironment` with a `VacuumState` that
    contains locations (graph adjacency), dirt distribution, agent location,
    and step counter. The environment evolves via its transition (action
    handlers) and does not depend on the agent implementation.

A (Actuators): modular `action_handlers` mapping action names (e.g.,
    Left/Right/Suck/NoOp) to handler callables. New actions can be added by
    adding entries to this mapping.

S (Sensors): modular `SensorModel` protocol. Example implementations include
    `LocalDirtSensor` and `NoisyLocalDirtSensor`. Percepts are produced by
    calling `env.percept()` which delegates to the sensor.
"""



# ============================================================
# Actuators (Actions) - extensible
# ============================================================

class Action:
    LEFT = "Left"
    RIGHT = "Right"
    SUCK = "Suck"
    NOOP = "NoOp"


# ============================================================
# Sensors (Percepts) - swappable / modular
# ============================================================

@dataclass(frozen=True)
class Percept:
    location: str
    is_dirty: bool


class SensorModel(Protocol):
    def sense(self, env: "VacuumEnvironment") -> Percept:
        ...


class LocalDirtSensor:
    """Percept = (current location, dirt status at current location)."""
    def sense(self, env: "VacuumEnvironment") -> Percept:
        loc = env.state.agent_location
        return Percept(location=loc, is_dirty=env.state.dirt.get(loc, False))


class NoisyLocalDirtSensor:
    """Example alternative sensor: flips dirt reading with probability flip_prob."""
    def __init__(self, flip_prob: float = 0.1):
        self.flip_prob = flip_prob

    def sense(self, env: "VacuumEnvironment") -> Percept:
        loc = env.state.agent_location
        true_dirty = env.state.dirt.get(loc, False)
        observed_dirty = (not true_dirty) if random.random() < self.flip_prob else true_dirty
        return Percept(location=loc, is_dirty=observed_dirty)


# ============================================================
# Environment State (size/shape extensible)
# ============================================================

@dataclass
class VacuumState:
    locations: List[str]                 # e.g., ["A","B"] or more
    adjacency: Dict[str, List[str]]      # shape as a graph
    agent_location: str
    dirt: Dict[str, bool]                # dirt distribution
    step: int = 0


# ============================================================
# Performance Measure (tracks cumulative performance)
# ============================================================

class PerformanceEvaluator:
    """
    Example performance function from the textbook-style description:
      +1 per clean square per time step
      -1 per action per time step
    Easily adjustable.
    """

    def __init__(self, reward_clean_per_square: int = 1, action_cost: int = 1):
        self.reward_clean_per_square = reward_clean_per_square
        self.action_cost = action_cost
        self.cumulative_score = 0

    def score_step(self, state: VacuumState, action: str) -> int:
        clean_squares = sum(1 for loc in state.locations if not state.dirt.get(loc, False))
        delta = (self.reward_clean_per_square * clean_squares) - self.action_cost
        self.cumulative_score += delta
        return delta


# ============================================================
# Environment Dynamics (transition function) - independent
# ============================================================

class VacuumEnvironment:
    """
    Modular, performance-measuring simulator for the vacuum world.

    Key property: agent program is separate.
    The environment provides percepts and applies actions via a transition function.
    """

    def __init__(
        self,
        initial_state: VacuumState,
        sensor: SensorModel,
        evaluator: PerformanceEvaluator,
        action_handlers: Optional[Dict[str, Callable[["VacuumEnvironment"], None]]] = None,
    ):
        self.state = initial_state
        self.sensor = sensor
        self.evaluator = evaluator

        # Modular actuators: action -> handler mapping (easy to extend)
        self.action_handlers = action_handlers or {
            Action.LEFT: self._handle_left,
            Action.RIGHT: self._handle_right,
            Action.SUCK: self._handle_suck,
            Action.NOOP: self._handle_noop,
        }

    def percept(self) -> Percept:
        """Generate a percept for the agent (modular sensor model)."""
        return self.sensor.sense(self)

    def step(self, action: str) -> Tuple[VacuumState, int]:
        """
        Apply one action using the environment transition function.
        Returns (new_state, performance_delta_for_this_step).
        """
        if action not in self.action_handlers:
            raise ValueError(f"Unknown action: {action}")

        # Apply action (environment changes itself)
        self.action_handlers[action](self)

        # Advance time
        self.state.step += 1

        # Update performance (environment tracks it)
        delta = self.evaluator.score_step(self.state, action)
        return self.state, delta

    def run(self, agent_program: "AgentProgram", steps: int) -> int:
        """
        Run a simulation episode for a fixed number of steps.
        Returns cumulative performance score.
        """
        for _ in range(steps):
            p = self.percept()
            a = agent_program(p)      # agent maps percept -> action
            self.step(a)              # environment applies action
        return self.evaluator.cumulative_score

    # ---------- Default action handlers ----------
    # Written to work for any adjacency graph.

    def _handle_left(self, _: "VacuumEnvironment") -> None:
        neighbors = self.state.adjacency.get(self.state.agent_location, [])
        if neighbors:
            self.state.agent_location = neighbors[0]

    def _handle_right(self, _: "VacuumEnvironment") -> None:
        neighbors = self.state.adjacency.get(self.state.agent_location, [])
        if neighbors:
            self.state.agent_location = neighbors[-1]

    def _handle_suck(self, _: "VacuumEnvironment") -> None:
        self.state.dirt[self.state.agent_location] = False

    def _handle_noop(self, _: "VacuumEnvironment") -> None:
        pass


# ============================================================
# Agent program interface (kept separate from environment)
# ============================================================

class AgentProgram(Protocol):
    def __call__(self, percept: Percept) -> str:
        ...


# Simple demo agent (optional): classic reflex
class SimpleReflexAgent:
    def __call__(self, percept: Percept) -> str:
        if percept.is_dirty:
            return Action.SUCK
        return Action.RIGHT


# Random agent: chooses uniformly among available primitive actions.
class RandomAgent:
    def __init__(self, seed: Optional[int] = None):
        self.rng = random.Random(seed)

    def __call__(self, percept: Percept) -> str:
        return self.rng.choice([Action.SUCK, Action.LEFT, Action.RIGHT, Action.NOOP])


# ============================================================
# World builder utilities (size/shape/dirt placement configurable)
# ============================================================

def build_two_location_world(
    dirt_A: bool = True,
    dirt_B: bool = True,
    start: str = "A",
) -> VacuumState:
    locations = ["A", "B"]
    adjacency = {"A": ["B"], "B": ["A"]}  # A <-> B
    dirt = {"A": dirt_A, "B": dirt_B}
    return VacuumState(locations=locations, adjacency=adjacency, agent_location=start, dirt=dirt)


def build_line_world(n: int, dirt_prob: float = 0.5, start_index: int = 0, seed: int = 0) -> VacuumState:
    """
    Example extensibility: a line of n locations (L0-L{n-1}).
    'Left' moves to previous neighbor, 'Right' moves to next neighbor.
    Dirt placement is random with probability dirt_prob.
    """
    rng = random.Random(seed)
    locations = [f"L{i}" for i in range(n)]
    adjacency: Dict[str, List[str]] = {}
    for i, loc in enumerate(locations):
        neighbors = []
        if i - 1 >= 0:
            neighbors.append(locations[i - 1])
        if i + 1 < n:
            neighbors.append(locations[i + 1])
        adjacency[loc] = neighbors

    dirt = {loc: (rng.random() < dirt_prob) for loc in locations}
    start = locations[max(0, min(start_index, n - 1))]
    return VacuumState(locations=locations, adjacency=adjacency, agent_location=start, dirt=dirt)


# ============================================================
# Demo main (shows it runs + prints cumulative performance)
# ============================================================

def main() -> None:
    # Minimum required world: two locations
    state = build_two_location_world(dirt_A=True, dirt_B=False, start="A")

    # Modular components:
    sensor = LocalDirtSensor()  # swap to NoisyLocalDirtSensor(...) if you want
    evaluator = PerformanceEvaluator(reward_clean_per_square=1, action_cost=1)

    env = VacuumEnvironment(initial_state=state, sensor=sensor, evaluator=evaluator)

    # Demo run with a simple reflex agent
    agent = SimpleReflexAgent()
    total = env.run(agent, steps=10)

    print("=== Vacuum World Simulator (Exercise 2.11) ===")
    print("Final state:", env.state)
    print("Cumulative performance (simple reflex, single run):", total)

    # Experimental requirement: run at least two different agents (reflex vs random)
    def run_trials(agent_factory: Callable[[], AgentProgram], trials: int = 100, steps: int = 30):
        results = []
        for t in range(trials):
            # Randomize initial dirt distribution for each trial (two-location world)
            dirtA = random.choice([True, False])
            dirtB = random.choice([True, False])
            state_t = build_two_location_world(dirt_A=dirtA, dirt_B=dirtB, start=random.choice(["A", "B"]))
            sensor_t = LocalDirtSensor()
            evaluator_t = PerformanceEvaluator(reward_clean_per_square=1, action_cost=1)
            env_t = VacuumEnvironment(initial_state=state_t, sensor=sensor_t, evaluator=evaluator_t)
            agent_t = agent_factory()
            score = env_t.run(agent_t, steps=steps)
            results.append(score)
        return results

    trials = 100
    steps = 30
    reflex_results = run_trials(lambda: SimpleReflexAgent(), trials=trials, steps=steps)
    random_results = run_trials(lambda: RandomAgent(), trials=trials, steps=steps)

    print(f"\nExperimental comparison over {trials} trials, {steps} steps each:")
    print(f"SimpleReflexAgent: mean={statistics.mean(reflex_results):.2f}, stdev={statistics.pstdev(reflex_results):.2f}")
    print(f"RandomAgent:      mean={statistics.mean(random_results):.2f}, stdev={statistics.pstdev(random_results):.2f}")

if __name__ == "__main__":
    main()

=== Vacuum World Simulator (Exercise 2.11) ===
Final state: VacuumState(locations=['A', 'B'], adjacency={'A': ['B'], 'B': ['A']}, agent_location='B', dirt={'A': False, 'B': False}, step=10)
Cumulative performance (simple reflex, single run): 10

Experimental comparison over 100 trials, 30 steps each:
SimpleReflexAgent: mean=29.29, stdev=0.83
RandomAgent:      mean=21.52, stdev=8.55


In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Tuple, Set, Optional, Protocol, List, Callable
import random
import statistics


# ============================================================
# Actions
# ============================================================

class Action:
    UP = "Up"
    DOWN = "Down"
    LEFT = "Left"
    RIGHT = "Right"
    SUCK = "Suck"
    NOOP = "NoOp"


# ============================================================
# Percepts 
# ============================================================

@dataclass(frozen=True)
class Percept:
    location: Tuple[int, int]
    is_dirty: bool
    bump: bool  # True if last movement action failed (wall/obstacle)


class AgentProgram(Protocol):
    def __call__(self, percept: Percept) -> str:
        ...


# ============================================================
# Environment state
# ============================================================

@dataclass
class GridState:
    width: int
    height: int
    obstacles: Set[Tuple[int, int]]
    dirt: Dict[Tuple[int, int], bool]
    agent_location: Tuple[int, int]
    step: int = 0


# ============================================================
# Performance measure
# ============================================================

class PerformanceEvaluator:
    def __init__(self, reward_clean_per_square: int = 1, action_cost: int = 1):
        self.reward_clean_per_square = reward_clean_per_square
        self.action_cost = action_cost
        self.cumulative_score = 0

    def score_step(self, state: GridState, action: str) -> int:
        clean_squares = sum(
            1
            for x in range(state.width)
            for y in range(state.height)
            if (x, y) not in state.obstacles and not state.dirt.get((x, y), False)
        )
        delta = (self.reward_clean_per_square * clean_squares) - self.action_cost
        self.cumulative_score += delta
        return delta


# ============================================================
# Environment dynamics
# ============================================================

class GridEnvironment:
    """Grid environment with unknown geography to the agent."""

    def __init__(self, initial_state: GridState, evaluator: PerformanceEvaluator):
        self.state = initial_state
        self.evaluator = evaluator
        self.last_bump: bool = False

    def percept(self) -> Percept:
        loc = self.state.agent_location
        return Percept(
            location=loc,
            is_dirty=self.state.dirt.get(loc, False),
            bump=self.last_bump,
        )

    def step(self, action: str) -> Tuple[GridState, int]:
        x, y = self.state.agent_location
        new_loc = (x, y)
        self.last_bump = False  # reset each step

        if action == Action.UP:
            candidate = (x, y - 1)
            if self._is_free(candidate):
                new_loc = candidate
            else:
                self.last_bump = True
        elif action == Action.DOWN:
            candidate = (x, y + 1)
            if self._is_free(candidate):
                new_loc = candidate
            else:
                self.last_bump = True
        elif action == Action.LEFT:
            candidate = (x - 1, y)
            if self._is_free(candidate):
                new_loc = candidate
            else:
                self.last_bump = True
        elif action == Action.RIGHT:
            candidate = (x + 1, y)
            if self._is_free(candidate):
                new_loc = candidate
            else:
                self.last_bump = True
        elif action == Action.SUCK:
            self.state.dirt[self.state.agent_location] = False
        elif action == Action.NOOP:
            pass
        else:
            raise ValueError(f"Unknown action: {action}")

        self.state.agent_location = new_loc
        self.state.step += 1
        delta = self.evaluator.score_step(self.state, action)
        return self.state, delta

    def _is_free(self, loc: Tuple[int, int]) -> bool:
        x, y = loc
        if x < 0 or y < 0 or x >= self.state.width or y >= self.state.height:
            return False
        if loc in self.state.obstacles:
            return False
        return True


# ============================================================
# World builder
# ============================================================

def build_random_grid(
    width: int,
    height: int,
    obstacle_prob: float,
    dirt_prob: float,
    seed: Optional[int] = None
) -> GridState:
    rng = random.Random(seed)
    obstacles: Set[Tuple[int, int]] = set()
    dirt: Dict[Tuple[int, int], bool] = {}

    for x in range(width):
        for y in range(height):
            if rng.random() < obstacle_prob:
                obstacles.add((x, y))
            else:
                dirt[(x, y)] = (rng.random() < dirt_prob)

    free_cells = [c for c in dirt.keys() if c not in obstacles]
    if not free_cells:
        raise ValueError("No free cells available")

    start = rng.choice(free_cells)
    return GridState(
        width=width,
        height=height,
        obstacles=obstacles,
        dirt=dirt,
        agent_location=start,
    )


# ============================================================
# Agents
# ============================================================

class SimpleReflexAgent:
    """Simple reflex: if dirty -> SUCK else always move RIGHT (blind)."""
    def __call__(self, percept: Percept) -> str:
        if percept.is_dirty:
            return Action.SUCK
        return Action.RIGHT


class RandomizedReflexAgent:
    """If dirty -> SUCK else pick a random move."""
    def __init__(self, seed: Optional[int] = None):
        self.rng = random.Random(seed)

    def __call__(self, percept: Percept) -> str:
        if percept.is_dirty:
            return Action.SUCK
        return self.rng.choice([Action.UP, Action.DOWN, Action.LEFT, Action.RIGHT, Action.NOOP])


class StatefulReflexAgent:
    """
    Reflex agent with state:
    - Uses bump feedback to learn blocked directions from specific cells.
    - Tracks visited cells and prefers unvisited neighbors first.
    """

    DELTAS = {
        Action.UP: (0, -1),
        Action.DOWN: (0, 1),
        Action.LEFT: (-1, 0),
        Action.RIGHT: (1, 0),
    }

    def __init__(self):
        self.known_free: Set[Tuple[int, int]] = set()
        self.blocked: Dict[Tuple[int, int], Set[str]] = {}
        self.visited: Set[Tuple[int, int]] = set()

        self.last_location: Optional[Tuple[int, int]] = None
        self.last_action: Optional[str] = None

    def __call__(self, percept: Percept) -> str:
        loc = percept.location
        self.known_free.add(loc)
        self.visited.add(loc)

        # Learns blocked transitions from bump feedback.
        if percept.bump and self.last_location is not None and self.last_action in self.DELTAS:
            self.blocked.setdefault(self.last_location, set()).add(self.last_action)

        if percept.is_dirty:
            self.last_location = loc
            self.last_action = Action.SUCK
            return Action.SUCK

        # Prefers unvisited neighbors
        for a, (dx, dy) in self.DELTAS.items():
            if a in self.blocked.get(loc, set()):
                continue
            cand = (loc[0] + dx, loc[1] + dy)
            if cand not in self.visited:
                self.last_location = loc
                self.last_action = a
                return a

        # Otherwise, try any unblocked move
        for a in self.DELTAS:
            if a in self.blocked.get(loc, set()):
                continue
            self.last_location = loc
            self.last_action = a
            return a

        self.last_location = loc
        self.last_action = Action.NOOP
        return Action.NOOP


# ============================================================
# Experiments
# ============================================================

def run_episode(env: GridEnvironment, agent: AgentProgram, steps: int) -> int:
    for _ in range(steps):
        p = env.percept()
        a = agent(p)
        env.step(a)
    return env.evaluator.cumulative_score


def experiment_random_environments(
    num_envs: int = 50,
    width: int = 6,
    height: int = 6,
    obstacle_prob: float = 0.1,
    dirt_prob: float = 0.2,
    steps: int = 200,
):
    rng = random.Random(0)

    # use factories to create a fresh agent per environment/run
    agent_factories: List[Tuple[str, Callable[[], AgentProgram]]] = [
        ("SimpleReflex", lambda: SimpleReflexAgent()),
        ("RandomizedReflex", lambda: RandomizedReflexAgent()),
        ("StatefulReflex", lambda: StatefulReflexAgent()),
    ]

    results: Dict[str, List[float]] = {name: [] for name, _ in agent_factories}

    for _ in range(num_envs):
        state = build_random_grid(width, height, obstacle_prob, dirt_prob, seed=rng.randint(0, 10**9))

        for name, make_agent in agent_factories:
            evaluator = PerformanceEvaluator(reward_clean_per_square=1, action_cost=1)

            # Copy state so each agent faces the same environment
            state_copy = GridState(
                width=state.width,
                height=state.height,
                obstacles=set(state.obstacles),
                dirt=dict(state.dirt),
                agent_location=state.agent_location,
            )
            env = GridEnvironment(initial_state=state_copy, evaluator=evaluator)
            agent = make_agent()  # fresh instance each run

            score = run_episode(env, agent, steps)
            results[name].append(score)

    print(f"\nExperiment: {num_envs} random environments, {steps} steps each")
    for name, vals in results.items():
        print(f"{name}: mean={statistics.mean(vals):.2f}, stdev={statistics.pstdev(vals):.2f}")


def experiment_corridor_poor_for_random(length: int = 10, dirt_at_end: bool = True, steps: int = 200):
    # 1 x length corridor: start at (0,0) and dirt at far right
    width = length
    height = 1
    obstacles: Set[Tuple[int, int]] = set()
    dirt: Dict[Tuple[int, int], bool] = {(x, 0): False for x in range(width)}
    if dirt_at_end:
        dirt[(width - 1, 0)] = True

    state = GridState(width=width, height=height, obstacles=obstacles, dirt=dirt, agent_location=(0, 0))

    agents: List[Tuple[str, Callable[[], AgentProgram]]] = [
        ("RandomizedReflex", lambda: RandomizedReflexAgent(seed=1)),
        ("StatefulReflex", lambda: StatefulReflexAgent()),
        ("SimpleReflex", lambda: SimpleReflexAgent()),
    ]

    print(f"\nCorridor experiment length={length}, dirt_at_end={dirt_at_end}, steps={steps}")
    for name, make_agent in agents:
        evaluator = PerformanceEvaluator(reward_clean_per_square=1, action_cost=1)
        state_copy = GridState(
            width=state.width,
            height=state.height,
            obstacles=set(state.obstacles),
            dirt=dict(state.dirt),
            agent_location=state.agent_location,
        )
        env = GridEnvironment(initial_state=state_copy, evaluator=evaluator)
        agent = make_agent()
        score = run_episode(env, agent, steps)
        print(f"{name}: score={score}")


def main():
    print("=== Vacuum World (Exercise 2.14) experiments ===")
    experiment_random_environments(
        num_envs=40,
        width=6,
        height=6,
        obstacle_prob=0.08,
        dirt_prob=0.12,
        steps=200,
    )
    experiment_corridor_poor_for_random(length=12, dirt_at_end=True, steps=300)


if __name__ == "__main__":
    main()

=== Vacuum World (Exercise 2.14) experiments ===

Experiment: 40 random environments, 200 steps each
SimpleReflex: mean=5734.00, stdev=504.60
RandomizedReflex: mean=6101.30, stdev=384.34
StatefulReflex: mean=6098.57, stdev=455.79

Corridor experiment length=12, dirt_at_end=True, steps=300
RandomizedReflex: score=3234
StatefulReflex: score=3266
SimpleReflex: score=3289
